# Sesión 3 · Robusto: una pregunta, muchos análisis

**Objetivo.** Medir cuánto cambia la respuesta cuando el agente vuelve a correr o elige otro método
razonable, y escribir una regla de robustez en la especificación. **Tiempo:** 80 minutos.

In [ ]:
# --- Configuración: correr una vez al empezar ---
import os, sys, json
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
import pandas as pd
import matplotlib.pyplot as plt
from agente.llm import desde_entorno
from agente.harness import ejecutar_agente, correr_varias, leer_traza, con_especificacion, SISTEMA_BASE
from agente.evaluacion import tarjeta, huerfanos, guardar, cargar, cargar_respuesta, diagnosticos_ejecutados

# os.environ["LLM_PROVEEDOR"] = "ollama"   # o "anthropic" / "openai"; si no lo configurás, se usa el simulador
# os.environ["LLM_MODELO"] = "qwen3.5:4b"   # el modelo que descargaste (ver INSTALACION.md)
llm = desde_entorno()
df = pd.read_csv(RAIZ / "datos" / "programa_x.csv")
TRAZAS = RAIZ / "trazas" / "s3"
RES = RAIZ / "resultados"; RES.mkdir(exist_ok=True)
VERDAD, TOLERANCIA = cargar_respuesta(RAIZ)
PREGUNTA = "¿Cuánto aumentó el programa el ingreso posterior de quienes participaron?"
ESPEC = (RAIZ / "especificaciones" / "especificacion.md").read_text(encoding="utf-8")
SISTEMA = con_especificacion(ESPEC)

## 1. Veinte corridas, la misma instrucción (20 min)

In [ ]:
res20 = correr_varias(20, PREGUNTA, df, llm, sistema=SISTEMA, carpeta_trazas=TRAZAS, etiqueta="s3")
guardar(res20, RES / "s3_20.json")
res20.metodo.value_counts()

In [ ]:
v = res20.dropna(subset=["estimacion"])
fig, ax = plt.subplots(figsize=(9, 2.5))
ax.scatter(v.estimacion, pd.Series(range(len(v))) % 5, alpha=.7)
if VERDAD:
    ax.axvline(VERDAD, color="k", lw=1)
    ax.axvspan(VERDAD - TOLERANCIA, VERDAD + TOLERANCIA, color="k", alpha=.08)
ax.set_yticks([]); ax.set_xlabel("estimación (pesos del programa)")
ax.set_title("20 corridas con la misma instrucción · línea: efecto experimental")
plt.show()

## 2. El multiverso: imponer cada método (25 min)

En vez de dejar que el agente elija, le imponemos el método. Así vemos cuánto depende la respuesta
de esa sola elección (Steegen et al., 2016; Simonsohn et al., 2020).

In [ ]:
METODOS = {"dif_medias": "diferencia de medias simple",
           "mco": "regresión lineal con covariables",
           "emparejamiento": "emparejamiento 1 a 1 por propensión",
           "ipw": "ponderación por propensión inversa",
           "doble_robusto": "estimador doble robusto (AIPW)"}
partes = []
for m, desc in METODOS.items():
    r = correr_varias(3, PREGUNTA, df, llm, sistema=SISTEMA + f"\nMÉTODO OBLIGATORIO: {m} ({desc})",
                      carpeta_trazas=TRAZAS, etiqueta=f"s3-{m}")
    r["metodo_forzado"] = m
    partes.append(r)
multiverso = pd.concat(partes, ignore_index=True)
guardar(multiverso, RES / "s3_multiverso.json")

In [ ]:
v = multiverso.dropna(subset=["estimacion"]).sort_values("estimacion").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9, 4))
for m, g in v.groupby("metodo_forzado"):
    abajo = (g.estimacion - g.ic_inferior).fillna(0)
    arriba = (g.ic_superior - g.estimacion).fillna(0)
    ax.errorbar(g.index, g.estimacion, yerr=[abajo, arriba], fmt="o", capsize=2, label=m)
if VERDAD:
    ax.axhline(VERDAD, color="k", lw=1)
    ax.axhspan(VERDAD - TOLERANCIA, VERDAD + TOLERANCIA, color="k", alpha=.08)
ax.set_xlabel("especificaciones, ordenadas por estimación"); ax.set_ylabel("estimación")
ax.set_title("Curva de especificación"); ax.legend(fontsize=8)
plt.show()

## 3. ¿Cada cuánto acierta? pass^k (15 min)

**pass^1** es la probabilidad de que una corrida al azar acierte. **pass^k** es la probabilidad de que
*k* corridas acierten **todas** (Yao et al., 2024). Si el agente va a decidir sin supervisión,
lo que importa es pass^k.

In [ ]:
tarjeta(res20, VERDAD, TOLERANCIA, ks=(1, 3, 5, 10))

## 4. Una regla de robustez (15 min)

Agregá a tu especificación una regla. Por ejemplo: «Estimá con al menos tres métodos distintos y
reportá todos; si no coinciden dentro de un margen que vos fijes, decilo y no elijas uno». Guardá el
archivo, volvé a cargarlo y corré cinco veces.

In [ ]:
ESPEC = (RAIZ / "especificaciones" / "especificacion.md").read_text(encoding="utf-8")
SISTEMA = con_especificacion(ESPEC)
res_regla = correr_varias(5, PREGUNTA, df, llm, sistema=SISTEMA, carpeta_trazas=TRAZAS, etiqueta="s3-regla")
guardar(res_regla, RES / "s3_regla.json")
tarjeta(res20, VERDAD, TOLERANCIA).merge(tarjeta(res_regla, VERDAD, TOLERANCIA),
                                         on=["pilar", "métrica"], suffixes=(" · 20 corridas", " · con regla"))

## 5. ¿Estás haciendo p-hacking? (5 min)

Ya conocés la respuesta. Si retocás la especificación hasta que el agente acierte, la estás ajustando
a esta muestra: es p-hacking con otro nombre (Thomas, Gligoric & Shah, 2026). Por eso la evaluación
final de la sesión 5 usa datos que todavía no viste.

## Para la próxima

- **Leer:** Imbens & Xu (2025), sobre este mismo programa; Heusser et al. (2026), «Open-Sourcing Decision Lab».